# Loading dataset



In [ ]:
import pandas as pd

train_label = pd.read_csv("data-circle/projects/water_pumps/team-WTM/data/raw/Training_set_labels.csv")

train_values = pd.read_csv("data-circle/projects/water_pumps/team-WTM/data/raw/Training_set_values.csv")

dataset dimensions and structure

In [ ]:
print("train_values_shape:", train_values.shape)
print("train_label_shape:", train_label.shape)

In [ ]:
train_values.info()

In [ ]:
train_label.info()

In [ ]:
print("Unique IDs in labels:", train_label["id"].nunique())
print("Unique IDs in values:", train_values["id"].nunique())

Merging dataset

In [ ]:
train_data = train_values.merge(
    train_label,
    how="outer",
    on="id",
    sort=True,
    indicator=True
)

In [ ]:
train_data["_merge"].value_counts()

train_data = train_data.drop(columns="_merge")

In [ ]:
# checking the structure of the merged dataset

train_data.info()

In [ ]:
#checking the dimension of merged dataset

train_data.shape

In [ ]:
train_data.head(10)

In [ ]:
train_data.dtypes

In [ ]:
rawdata_dictionary = pd.DataFrame({
    "Column": train_data.columns,
    "Data Type": train_data.dtypes.astype(str).values,
    "Missing Values": train_data.isnull().sum().values,
    "Unique Values": train_data.nunique().values
})

rawdata_dictionary

# Data Quality Checks

In [ ]:
#1. Missing Values
missing = train_data.isnull().sum().sort_values(ascending=False)

print(missing)

From the missing value check about eight columns have missing values

In [ ]:
#2. Checking the distribution of the target variable:
train_data["status_group"].value_counts()

This tells how balanced the target variable are, but in this case, the dataset is not perfectly balanced. 
The functional needs repair category is smaller than the other two. This can affect machine learning performance.

In [ ]:
#3.Checking unique values in categorical columns.
categorical_columns = train_data.select_dtypes(include=["str"]).columns.tolist()

categorical_columns.remove("date_recorded")

print(categorical_columns)

NB: dtae_recorded is excluded from the categorical inspection because it is a date and not a true categorical variable

In [ ]:
#To visualise the uniques values in the categorical variables

for column in categorical_columns:
    print("\n", column)
    print(train_data[column].unique())

Note: From observation, some feature variables contain related information but in some cases more broader classification as seen above E.g source and source_type.
However,some column contain excatly the same infromation for every row: quantity and quantity_group. 

Also to further conclude that the columns are dentical, checking whether every row hass exactly the same vaue in both columns is necessary. 
(train_data["quantity"] == train_data["quantity_group"]).all()

Feature selection: Related or grouped variables will be assessed for redundancy and missingness. Where variables contained  highly overlapping information, the variable with lower missingness will be retained and the other ignored in the modelling.

Features that are very related:

Funder              → installer

source              → source_type

quantity            → quantity_group

water_quality       → quality_group

extraction_type     → extraction_type_group 

waterpoint_type     → waterpoint_type_group

management           → management_group

In [ ]:
#4. identifying the high-cardinality variables
#Cardinality_summary

cardinality_summary = pd.DataFrame({
    "Unique Values": train_data[categorical_columns].nunique(),
    "Missing Values": train_data[categorical_columns].isna().sum()
}).sort_values("Unique Values", ascending=False)

cardinality_summary

Numerical columns: To identify suspicious or invalid values



In [ ]:
numeric_columns = train_data.select_dtypes(include=["int64", "float64"]).columns

print(numeric_columns)

In [ ]:
Note : 1. gps_height can be zero or negative

        If zero it means the waterpoint is located approximately at sea level

        Negative would mean the location is below sea level.

        2. #num_private column can be removed because it identify invalid  values: mostly zeros

In [ ]:
# Display their data types
print(train_data[numeric_columns].dtypes)


In [ ]:
# Statistical summary
train_data[numeric_columns].describe()

In [ ]:
numeric_missing = pd.DataFrame({
    "Missing Values": train_data[numeric_columns].isna().sum(),
})

numeric_missing

In [ ]:
#an overview of the numeric_columns
train_data[numeric_columns].head(10)

In [ ]:
Observation: 1.No missing values in the numeric_columns
            2.population and construction year should be investigated, 
            as their are 0 values whicha are invalid and showuld be handled differently.
            We can replace the 0 with missing or unknown

In [ ]:
import numpy as np

train_data["population"] = train_data["population"].replace(0, np.nan)

train_data["construction_year"] = train_data["construction_year"].replace(0, np.nan)

In [ ]:
# convert date_recorded from string to datatime type
train_data["date_recorded"] = pd.to_datetime(train_data["date_recorded"])


date_recorded was initially stored as a string/object data type and should be converted to datetime format 
for temporal analysis and feature engineering.